# Модуль 4 — Прогнозирование возможного отставания (baseline)


**Контракт модуля** (см. архитектуру системы):

- **Вход:**
  - результат модуля 3 (определение текущей фазы) — в baseline представлен контрактом:
    ```json
    {
      "project_id": "...",
      "status_date": "YYYY-MM-DD",
      "current_phase": "...",
      "current_phase_order": 5,
      "phase_confidence": 0.93,
      "current_phase_started_at": "YYYY-MM-DD"
    }
    ```
  - канонический план проекта (даты/длительности проекта и текущей фазы)
  - текущая/заданная дата статуса
- **Выход:** прогноз отставания (`DelayForecast`) — статус, оценка отставания на сегодня, прогнозируемое отставание к финишу, прогнозная дата завершения, эффективный SPI(t), причина.

Важно: `current_phase_started_at` должен приходить из истории фаз (модуль 3), а не выдумываться модулем 4.

**Математика baseline:**

1. Прогресс внутри текущей фазы: `phase_progress = days_in_phase / phase_planned_duration_days` (обрезается в [0, 1]).
2. Earned Schedule: `ES = phase_start_offset_days + phase_progress * phase_planned_duration_days`.
3. Schedule Performance Index по времени: `SPI(t) = ES / actual_time`.
4. Сжатие по доверию к фазе (`phase_confidence` от модуля 3): `effective_SPI = 1 + confidence * (SPI(t) - 1)` — при низком доверии прогноз стягивается к нейтральному SPI = 1.
5. Прогноз длительности проекта: `forecast_duration = planned_duration_days / effective_SPI`, отсюда прогнозное отставание и дата финиша.


## 1. Импорт baseline-логики


In [ ]:
from dataclasses import dataclass
from datetime import date
from typing import Optional


@dataclass
class DelayForecast:
    status: str
    estimated_delay_to_date_days: Optional[float]
    forecast_delay_days: Optional[float]
    forecast_finish_date: Optional[date]
    effective_spi_time: Optional[float]
    confidence: float
    reason: str


def forecast_delay(
    *,
    planned_start: date,
    planned_duration_days: float,
    phase_start_offset_days: float,
    phase_planned_duration_days: float,
    current_phase_started_at: date,
    status_date: date,
    phase_confidence: float = 1.0,
    warning_threshold_days: float = 7.0,
    critical_threshold_days: float = 30.0,
) -> DelayForecast:

    if status_date < planned_start:
        return DelayForecast(
            status="not_started",
            estimated_delay_to_date_days=None,
            forecast_delay_days=None,
            forecast_finish_date=None,
            effective_spi_time=None,
            confidence=phase_confidence,
            reason="Status date is before planned project start."
        )

    actual_time = (status_date - planned_start).days
    days_in_phase = max(0, (status_date - current_phase_started_at).days)

    if actual_time <= 0 or phase_planned_duration_days <= 0:
        return DelayForecast(
            status="insufficient_data",
            estimated_delay_to_date_days=None,
            forecast_delay_days=None,
            forecast_finish_date=None,
            effective_spi_time=None,
            confidence=phase_confidence,
            reason="Insufficient elapsed time or invalid phase duration."
        )

    phase_progress = min(1.0, max(0.0, days_in_phase / phase_planned_duration_days))
    earned_schedule = phase_start_offset_days + phase_progress * phase_planned_duration_days
    spi_t = earned_schedule / actual_time

    # Confidence shrinkage:
    # confidence=1 -> полностью доверяем детектору фазы
    # confidence=0 -> не отклоняемся от нейтрального SPI=1
    effective_spi = 1.0 + phase_confidence * (spi_t - 1.0)

    if effective_spi <= 0:
        return DelayForecast(
            status="insufficient_data",
            estimated_delay_to_date_days=actual_time - earned_schedule,
            forecast_delay_days=None,
            forecast_finish_date=None,
            effective_spi_time=effective_spi,
            confidence=phase_confidence,
            reason="Unstable forecast; effective schedule index <= 0."
        )

    estimated_delay_to_date = actual_time - earned_schedule
    forecast_total_duration = planned_duration_days / effective_spi
    forecast_delay = forecast_total_duration - planned_duration_days
    forecast_finish = planned_start.fromordinal(
        planned_start.toordinal() + round(forecast_total_duration)
    )

    if forecast_delay >= critical_threshold_days:
        status = "critical_delay"
    elif forecast_delay >= warning_threshold_days:
        status = "warning_delay"
    elif forecast_delay <= -warning_threshold_days:
        status = "ahead"
    else:
        status = "on_track"

    return DelayForecast(
        status=status,
        estimated_delay_to_date_days=estimated_delay_to_date,
        forecast_delay_days=forecast_delay,
        forecast_finish_date=forecast_finish,
        effective_spi_time=effective_spi,
        confidence=phase_confidence,
        reason="Earned-Schedule-style mathematical forecast."
    )

print("Модуль forecast_delay готов к использованию.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta

pd.set_option("display.max_colwidth", None)

## 2. Тестовые сценарии

Задаём несколько типовых ситуаций «на входе от модуля 3», чтобы посмотреть, как baseline реагирует:

1. **on_track** — фаза идёт по графику.
2. **warning_delay** — фаза заметно затянулась, но не критично.
3. **critical_delay** — сильное отставание.
4. **ahead** — работы идут с опережением.
5. **low_confidence** — тот же сценарий отставания, что и №2, но модуль 3 указал низкую уверенность в фазе — прогноз должен «стянуться» к нейтральному.
6. **not_started** — дата статуса раньше планового старта проекта.
7. **insufficient_data** — некорректная длительность фазы (данные ещё не накоплены).


In [ ]:
PLANNED_START = date(2026, 1, 1)
PLANNED_DURATION_DAYS = 200        # общая плановая длительность проекта
PHASE_START_OFFSET_DAYS = 60       # смещение начала текущей фазы от старта проекта по плану
PHASE_PLANNED_DURATION_DAYS = 40   # плановая длительность текущей фазы

scenarios = [
    dict(
        name="on_track",
        current_phase_started_at=date(2026, 3, 1),
        status_date=date(2026, 3, 20),
        phase_confidence=1.0,
    ),
    dict(
        name="warning_delay",
        current_phase_started_at=date(2026, 3, 1),
        status_date=date(2026, 4, 15),
        phase_confidence=1.0,
    ),
    dict(
        name="critical_delay",
        current_phase_started_at=date(2026, 3, 1),
        status_date=date(2026, 5, 20),
        phase_confidence=1.0,
    ),
    dict(
        name="ahead",
        current_phase_started_at=date(2026, 2, 20),
        status_date=date(2026, 3, 5),
        phase_confidence=1.0,
    ),
    dict(
        name="warning_delay_low_confidence",
        current_phase_started_at=date(2026, 3, 1),
        status_date=date(2026, 4, 15),
        phase_confidence=0.3,
    ),
    dict(
        name="not_started",
        current_phase_started_at=date(2026, 3, 1),
        status_date=date(2025, 12, 1),
        phase_confidence=1.0,
    ),
]

rows = []
for sc in scenarios:
    result = forecast_delay(
        planned_start=PLANNED_START,
        planned_duration_days=PLANNED_DURATION_DAYS,
        phase_start_offset_days=PHASE_START_OFFSET_DAYS,
        phase_planned_duration_days=PHASE_PLANNED_DURATION_DAYS,
        current_phase_started_at=sc["current_phase_started_at"],
        status_date=sc["status_date"],
        phase_confidence=sc["phase_confidence"],
    )
    row = {"scenario": sc["name"], "status_date": sc["status_date"], "phase_confidence": sc["phase_confidence"]}
    row.update(result.__dict__)
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df

### 2.1 Отдельно: сценарий `insufficient_data`

Если плановая длительность фазы задана некорректно (например, 0 — данные ещё не собраны), baseline честно сообщает о нехватке данных, а не выдаёт число.

In [ ]:
bad_result = forecast_delay(
    planned_start=PLANNED_START,
    planned_duration_days=PLANNED_DURATION_DAYS,
    phase_start_offset_days=PHASE_START_OFFSET_DAYS,
    phase_planned_duration_days=0,   # некорректные/ещё не собранные данные
    current_phase_started_at=date(2026, 3, 1),
    status_date=date(2026, 3, 20),
    phase_confidence=1.0,
)
bad_result

## 3. Динамика прогноза во времени

Фиксируем сценарий отставания и смотрим, как прогноз (эффективный SPI и прогнозируемое отставание) меняется по мере того, как накапливаются данные внутри фазы — от начала фазы и до конца её плановой длительности.


In [ ]:
phase_started = date(2026, 3, 1)
status_dates = [phase_started + timedelta(days=d) for d in range(0, 61, 2)]

timeline_rows = []
for sd in status_dates:
    r = forecast_delay(
        planned_start=PLANNED_START,
        planned_duration_days=PLANNED_DURATION_DAYS,
        phase_start_offset_days=PHASE_START_OFFSET_DAYS,
        phase_planned_duration_days=PHASE_PLANNED_DURATION_DAYS,
        current_phase_started_at=phase_started,
        status_date=sd,
        phase_confidence=1.0,
    )
    timeline_rows.append({
        "status_date": sd,
        "effective_spi_time": r.effective_spi_time,
        "forecast_delay_days": r.forecast_delay_days,
        "status": r.status,
    })

timeline_df = pd.DataFrame(timeline_rows)
timeline_df.tail()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].plot(timeline_df["status_date"], timeline_df["effective_spi_time"], marker="o", markersize=3)
axes[0].axhline(1.0, color="gray", linestyle="--", linewidth=1, label="SPI = 1 (график соблюдён)")
axes[0].set_ylabel("effective SPI(t)")
axes[0].set_title("Эффективный SPI(t) по мере накопления данных внутри фазы")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(timeline_df["status_date"], timeline_df["forecast_delay_days"], marker="o", markersize=3, color="tab:orange")
axes[1].axhline(0.0, color="gray", linestyle="--", linewidth=1)
axes[1].axhline(7.0, color="gold", linestyle=":", linewidth=1, label="warning_threshold")
axes[1].axhline(30.0, color="red", linestyle=":", linewidth=1, label="critical_threshold")
axes[1].set_ylabel("Прогноз отставания, дни")
axes[1].set_xlabel("Дата статуса")
axes[1].set_title("Прогнозируемое отставание проекта к финишу")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 4. Чувствительность прогноза к `phase_confidence`

Модуль 3 может быть не уверен в определении фазы. Смотрим, как один и тот же «сырой» сигнал отставания сжимается к нейтральному прогнозу при снижении доверия.


In [ ]:
confidence_levels = [1.0, 0.75, 0.5, 0.25, 0.0]
status_date_fixed = date(2026, 4, 15)  # тот же момент, что и в сценарии warning_delay

conf_rows = []
for c in confidence_levels:
    r = forecast_delay(
        planned_start=PLANNED_START,
        planned_duration_days=PLANNED_DURATION_DAYS,
        phase_start_offset_days=PHASE_START_OFFSET_DAYS,
        phase_planned_duration_days=PHASE_PLANNED_DURATION_DAYS,
        current_phase_started_at=date(2026, 3, 1),
        status_date=status_date_fixed,
        phase_confidence=c,
    )
    conf_rows.append({
        "phase_confidence": c,
        "effective_spi_time": r.effective_spi_time,
        "forecast_delay_days": r.forecast_delay_days,
        "status": r.status,
    })

conf_df = pd.DataFrame(conf_rows)
conf_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(conf_df["phase_confidence"], conf_df["forecast_delay_days"], marker="o")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("phase_confidence (доверие модуля 3 к текущей фазе)")
ax.set_ylabel("Прогноз отставания, дни")
ax.set_title("Сжатие прогноза к нейтральному SPI=1 при снижении доверия")
ax.grid(alpha=0.3)
plt.show()

## 5. Ручное тестирование произвольного сценария

Меняйте параметры ниже и перезапускайте ячейку — удобно для быстрой проверки конкретного кейса из реального проекта (например, полученного от модуля 3 через контракт JSON).


In [ ]:
manual_input = {
    "planned_start": date(2026, 1, 1),
    "planned_duration_days": 200,
    "phase_start_offset_days": 60,
    "phase_planned_duration_days": 40,
    "current_phase_started_at": date(2026, 3, 10),
    "status_date": date(2026, 4, 1),
    "phase_confidence": 0.85,
}

manual_result = forecast_delay(**manual_input)
manual_result

## 6. Итоги и заметки для интеграции

- Модуль 4 полностью детерминирован и не требует обучения — это чистая математика (Earned Schedule + confidence shrinkage), что удобно для baseline и объяснимости перед экспертом.
- Единственная зависимость от модуля 3 — поля `current_phase_started_at`, `current_phase_order`/`current_phase` (для сопоставления с планом) и `phase_confidence`. Эти поля должны приходить из истории фаз, а не переопределяться внутри модуля 4.
- Статусы `not_started` и `insufficient_data` — сигнал для оркестратора системы, что прогноз пока не может быть построен (например, проект ещё не начался или фаза только определена и данных внутри неё недостаточно).
- Пороги `warning_threshold_days` / `critical_threshold_days` — конфигурируемые параметры, их можно вынести в канонический формат плана или в настройки проекта.
- Дальнейшее развитие: замена постоянного `phase_planned_duration_days`/`phase_start_offset_days` на реальные значения из канонического плана (модуль 1) для каждой фазы конкретного проекта, а также учёт истории смены фаз (несколько фаз пройдено — модуль 4 может суммировать ES по всем завершённым фазам, а не только по текущей).
